# 00 · Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/00-quickstart.ipynb)

Welcome to the **Base Batches × Venice** workshop. This first notebook gets you connected to the Venice API and verifies everything works.

**What you'll learn**
1. How to set up your Venice API key (locally or in Colab).
2. How to make your first call.
3. How to list available models.

Venice is **OpenAI-compatible** — same SDK, just point `base_url` at `https://api.venice.ai/api/v1`. If you've ever used the OpenAI SDK, you already know Venice.

**Get your key:** https://venice.ai/settings/api

In [ ]:
%pip install --quiet openai requests python-dotenv rich

## 1. Connect

Three options for your API key:
- **Colab:** Click the 🔑 icon on the left sidebar → "+ Add new secret" → name it `VENICE_API_KEY`. Toggle "Notebook access" on.
- **Local:** Copy `.env.example` to `.env` and fill in your key.
- **Either:** The cell below will prompt you to paste it if it can't find one.

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 2. First call

If you've used `openai`, this is identical.

In [ ]:
resp = client.chat.completions.create(
    model="venice-uncensored",
    messages=[{"role": "user", "content": "In one sentence: why is privacy important for AI?"}],
)

print(resp.choices[0].message.content)

## 3. List the models

Venice has a *lot* of models — text, image, audio, video, embeddings. Let's see what's available.

In [ ]:
models = client.models.list()
print(f"Total models available: {len(models.data)}\n")

# Group by capability so it's easier to scan
from collections import defaultdict
by_type = defaultdict(list)
for m in models.data:
    spec = getattr(m, "model_spec", None) or {}
    if isinstance(spec, dict):
        mtype = spec.get("type", "unknown")
    else:
        mtype = getattr(spec, "type", "unknown")
    by_type[mtype].append(m.id)

for mtype, ids in sorted(by_type.items()):
    print(f"== {mtype} ({len(ids)}) ==")
    for mid in sorted(ids)[:8]:
        print(f"   {mid}")
    if len(ids) > 8:
        print(f"   ... and {len(ids) - 8} more")
    print()

## 4. Inspect Venice-specific response headers

Venice attaches a few extra headers that don't exist on OpenAI. Most useful:
- `x-venice-balance-usd` — your remaining credit
- `x-venice-balance-diem` — your DIEM-backed credit
- `x-venice-model-id` — which model actually handled the call (after routing)
- `x-ratelimit-remaining-*` — standard rate limit info

In [ ]:
import requests

r = requests.post(
    "https://api.venice.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    json={
        "model": "venice-uncensored",
        "messages": [{"role": "user", "content": "Say hi in 3 words."}],
    },
)

print(r.json()["choices"][0]["message"]["content"])
print()
print("Venice headers:")
for k, v in r.headers.items():
    if k.lower().startswith(("x-venice", "x-ratelimit")):
        print(f"  {k}: {v}")

## You're in 🎉

Next up:
- **[01 · Chat completions](./01-chat-completions.ipynb)** — streaming, system prompts, `venice_parameters`, web search, reasoning models
- **[02 · Embeddings + RAG](./02-embeddings-and-rag.ipynb)** — vector search on the Base Batches cohort
- **[07 · x402](./07-x402-wallet-payments.ipynb)** — pay for inference with a Base wallet, no API key
- **[08 · E2EE](./08-e2ee-encryption.ipynb)** — the cryptography behind privacy you can verify